# Data Cleaning

This notebook loads the extracted CSVs from `data/02_extraction/`, applies cleaning transformations via `COWCleaner`, and writes clean versions to `data/03_cleaning/`. Cleaning decisions are documented in `src/processing/cleaner.py`; this notebook demonstrates each step individually so intermediate state can be inspected.

## Implementation

Cleaning is implemented by `COWCleaner` in `src/processing/cleaner.py`. Each method is a discrete step that can be called individually for inspection.

| Method | Operation |
|---|---|
| `drop_event_columns()` | Remove spatially redundant and analytically irrelevant event columns |
| `deduplicate_events()` | Keep one row per warning at terminal status (`CAN > COR > EXP > EXT > CON > NEW`) |
| `parse_event_timestamps()` | Parse `issue`/`expire` to UTC datetimes; derive `duration_min` |
| `cap_lead0()` | Cap `lead0` at 99th percentile per phenomena; flag with `lead0_capped` |
| `derive_product_id()` | Add `product_id` join key (`{year}{wfo}{eventid}{phenomena}W1`) |
| `drop_stormreport_columns()` | Remove `type`, `magnitude`, `link` |
| `parse_stormreport_timestamps()` | Parse `valid` to UTC datetime |
| `normalize_source()` | Canonicalize source values; collapse automated stations; impute junk |
| `cap_leadtime()` | Cap `leadtime` at 99th percentile per lsrtype; flag with `leadtime_capped` |
| `impute_null_cities()` | Reverse-geocode null `city` values via Nominatim |
| `impute_null_counties()` | Reverse-geocode null `county` values via Nominatim |

In [1]:
import logging
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")

from cleaning import COWCleaner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)

# --- Configuration ---

# Input: flattened CSVs written by 02_extraction.ipynb
EXTRACTED_DIR = Path("../data/02_extraction")

# Output: cleaned, analysis-ready CSVs for downstream notebooks
PROCESSED_DIR = Path("../data/03_cleaning")

events       = pd.read_csv(EXTRACTED_DIR / "events.csv")
stormreports = pd.read_csv(EXTRACTED_DIR / "stormreports.csv")

print(f"events:       {len(events):,} rows × {len(events.columns)} columns")
print(f"stormreports: {len(stormreports):,} rows × {len(stormreports.columns)} columns")

cleaner = COWCleaner(processed_dir=PROCESSED_DIR)
print(cleaner)

events:       261,670 rows × 32 columns
stormreports: 378,288 rows × 19 columns
COWCleaner(processed=../data/03_cleaning)


## Events

### 1. Drop uninformative columns

Several columns add no analytical value and are dropped before any other work:

- `significance`: constant `'W'` (Warning) across all rows — no variance
- `perimeter`, `areaverify`, `lat0`, `lon0`: spatial geometry fields not used in statistical analysis
- `ar_ugc`, `ar_ugcname`: county/zone codes, not needed at this level of analysis
- `visual_imgurl`, `product_text`, `product_href`, `link`: URL/text blobs with no analytical use
- `stormreports`, `stormreports_all`: embedded JSON lists of storm report IDs; the `stormreports` table is the authoritative source for this relationship
- `fcster`: investigated as a staffing proxy but unusable — format varies within WFOs (badge numbers, last names, initials coexisting), making unique-count comparisons unreliable even within a single office
- `statuses`: identical to `status` after deduplication — a pre-dedup API artifact with no additional information

**Retained for regression (storm-mode covariates):**

- `svr_tornado_possible`: forecaster-flagged boolean on SV warnings indicating a tornado threat was simultaneously present — a direct indicator of prioritization pressure (the forecaster was watching both threats at once)
- `tor_in_svrtorpossible`: boolean indicating a tornado warning was active in the SVR tornado-possible area — a cleaner co-occurrence indicator than a time-window join
- `hailtag`, `windtag`: hail size (inches) and wind speed (knots) thresholds on SV warnings; partial proxies for storm intensity/mode
- `carea`, `parea`: warning polygon area and perimeter; larger warnings tend to be more speculative/precautionary
- `sharedborder`: shared border length with adjacent warnings; proxy for outbreak density

In [2]:
events = cleaner.drop_event_columns(events)
print(f"After drop: {len(events.columns)} columns remaining")
print(events.columns.tolist())

21:20:05  INFO     Dropped 15 event columns; 17 remaining


After drop: 17 columns remaining
['wfo', 'year', 'phenomena', 'eventid', 'sharedborder', 'issue', 'expire', 'svr_tornado_possible', 'hailtag', 'windtag', 'carea', 'parea', 'product_id', 'status', 'verify', 'tor_in_svrtorpossible', 'lead0']


### 2. Deduplicate events

The COW API returns one row per warning issuance status (`NEW`, `CON`, `EXT`, `EXP`, `CAN`, `COR`). For this analysis each warning is one event, so we keep only the terminal status row per `(wfo, year, phenomena, eventid)`. The terminal status is determined by priority: `CAN` > `COR` > `EXP` > `EXT` > `CON` > `NEW`. This preserves the final verified/cancelled state of each warning.

In [3]:
events = cleaner.deduplicate_events(events)
print(f"After deduplication: {len(events):,} rows")
print("\nStatus distribution after deduplication:")
print(events["status"].value_counts().to_string())

21:20:05  INFO     Deduplicated events: 261,670 → 261,557 rows (removed 113)


After deduplication: 261,557 rows

Status distribution after deduplication:
status
EXP    97124
CON    79785
CAN    53056
NEW    29814
EXT     1633
COR      145


### 3. Parse timestamps

`issue` and `expire` are ISO 8601 strings. Parse them to UTC-aware datetimes and derive `duration_min` (warning duration in minutes) as a convenience column.

In [4]:
events = cleaner.parse_event_timestamps(events)
print("issue range:",   events["issue"].min(),  "→", events["issue"].max())
print("expire range:",  events["expire"].min(), "→", events["expire"].max())
print("duration_min — min:", events["duration_min"].min(), "  max:", events["duration_min"].max())
events[["issue", "expire", "duration_min"]].head(3)

21:20:05  INFO     Parsed issue/expire timestamps; derived duration_min


issue range: 2016-01-01 00:36:00+00:00 → 2025-12-29 06:24:00+00:00
expire range: 2016-01-01 01:30:00+00:00 → 2025-12-29 07:00:00+00:00
duration_min — min: -8.0   max: 9224.0


,issue,expire,duration_min
0,2025-11-07 22:55:00+00:00,2025-11-07 23:39:00+00:00,44.0
1,2020-08-23 00:25:00+00:00,2020-08-23 01:33:00+00:00,68.0
2,2023-05-09 14:46:00+00:00,2023-05-09 14:57:00+00:00,11.0


### 4. Inspect `lead0`

`lead0` is the API-supplied lead time in minutes for the first confirming storm report for each verified warning event. It is already numeric in the extracted CSV — no parsing needed. Non-null for all verified events; null for unverified events.

In [5]:
verified = events[events["verify"] == True]
print(f"lead0 non-null: {verified['lead0'].notna().sum():,} of {len(verified):,} verified events")
print()
print(verified["lead0"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

lead0 non-null: 122,848 of 122,848 verified events

count    122848.000000
mean         21.191171
std          35.059846
min           0.000000
25%           5.000000
50%          13.000000
75%          25.000000
90%          43.000000
95%          65.000000
99%         167.000000
max        2768.000000


### 5. Cap extreme `lead0` values

FF has a max of ~7,911 minutes (~5.5 days), almost certainly from the API retroactively matching a warning to a storm report days later due to spatial overlap. TO and SV are naturally bounded. Cap per phenomena at the 99th percentile; `lead0_capped` flags affected rows.

In [6]:
events = cleaner.cap_lead0(events)
print(f"Total capped: {events['lead0_capped'].sum():,}")
print("\nPost-cap distribution by phenomena (verified only):")
verified_mask = events["verify"] == True
print(
    events[verified_mask]
    .groupby("phenomena")["lead0"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

21:20:05  INFO     lead0 cap: FF cap=323 min, 175 rows capped
21:20:05  INFO     lead0 cap: SV cap=52 min, 970 rows capped
21:20:05  INFO     lead0 cap: TO cap=41 min, 67 rows capped


Total capped: 1,212

Post-cap distribution by phenomena (verified only):
             count       mean        std  min   25%   50%   75%    95%    max
phenomena                                                                    
FF         17579.0  62.028272  62.299615  0.0  19.0  42.0  84.0  190.0  323.0
SV         98498.0  14.303468  12.393478  0.0   4.0  11.0  21.0   40.0   52.0
TO          6771.0  10.398907   9.789464  0.0   2.0   7.0  16.0   30.0   41.0


### 6. Derive join key (`product_id`)

The storm reports table links back to events via a VTEC product ID (e.g. `2020ABQ41SVW1`) stored in the `events` column. The format is `{year}{wfo}{eventid}{phenomena}W1`. Adding this as an explicit column on the events table makes the join unambiguous downstream.

In [7]:
events = cleaner.derive_product_id(events)

# Verify join key coverage
sr_event_ids = (
    stormreports[stormreports["warned"] == True]["events"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
)
match_rate = sr_event_ids.isin(events["product_id"]).mean()
print(f"SR → Event match rate via product_id: {match_rate:.2%}")
print("(Remaining misses have significance digit > 1 — co-issued warnings)")
events[["wfo", "year", "phenomena", "eventid", "product_id"]].head(3)

21:20:06  INFO     Derived product_id for 261,557 events


SR → Event match rate via product_id: 99.97%
(Remaining misses have significance digit > 1 — co-issued warnings)


,wfo,year,phenomena,eventid,product_id
0,OHX,2025,SV,268,2025OHX268SVW1
1,PBZ,2020,FF,21,2020PBZ21FFW1
2,GID,2023,TO,4,2023GID4TOW1


### 7. Save cleaned events

In [8]:
cleaner.save(events, "events.csv")
print(f"Saved {len(events):,} rows × {len(events.columns)} columns → data/03_cleaning/events.csv")
print(events.columns.tolist())

21:20:09  INFO     Saved 261,557 rows to ../data/03_cleaning/events.csv


Saved 261,557 rows × 19 columns → data/03_cleaning/events.csv
['wfo', 'year', 'phenomena', 'eventid', 'sharedborder', 'issue', 'expire', 'svr_tornado_possible', 'hailtag', 'windtag', 'carea', 'parea', 'product_id', 'status', 'verify', 'tor_in_svrtorpossible', 'lead0', 'duration_min', 'lead0_capped']


## Storm Reports

### 1. Repair malformed rows

A fixed-width field overflow bug in the IEM API corrupts adjacent columns when a long `remark` is serialized, affecting 26 rows across 10 WFOs. Four distinct patterns are repaired in order:

1. **lat0 truncation with clean duplicate** (18 rows) — the corrupt row is dropped; the clean copy is retained
2. **lat0 truncation without clean duplicate** (2 rows — MFL, SGF) — the missing leading digit is inferred from the plausible latitude range and prepended
3. **lon0 sign loss** (5 rows — ILX) — positive CONUS longitude repaired by negating; GUM (Guam/Saipan) with legitimately positive lon0 is excluded
4. **Junk state/county with valid coordinates** (1 row — PIH) — placeholder values nulled so the subsequent Nominatim imputation step resolves them

In [9]:
print(f"Rows before: {len(stormreports):,}")
stormreports = cleaner.repair_malformed_rows(stormreports)
print(f"Rows after:  {len(stormreports):,}")

Rows before: 378,288


21:20:10  WARNING  Dropped 20 corrupt rows with clean duplicates present
21:20:10  WARNING  Repaired lat0: idx=59132 wfo=CLE 1.02 → 41.02
21:20:10  WARNING  Repaired lat0: idx=247648 wfo=MFL 6.2 → 26.2
21:20:10  WARNING  Repaired lat0: idx=339770 wfo=SGF 7.84 → 37.84
21:20:10  WARNING  Repaired lon0 sign for 6 rows
21:20:10  WARNING  Nulled junk state/county for 11 rows


Rows after:  378,268


### 2. Drop uninformative columns

- `type`: single-character raw code that duplicates `lsrtype`; the latter is more readable
- `magnitude`: 67% null; magnitude values are not used in any planned analysis
- `link`: URL blob with no analytical use

`lon0` and `lat0` are retained for potential spatial EDA. `events` is retained as the trace link from each storm report back to its matched warning(s) — it is a comma-separated list of VTEC product IDs (e.g. `2020ABQ41SVW1`) and is null only for unwarned reports.

In [10]:
stormreports = cleaner.drop_stormreport_columns(stormreports)
print(f"After drop: {len(stormreports.columns)} columns remaining")
print(stormreports.columns.tolist())

21:20:10  INFO     Dropped 3 stormreport columns; 16 remaining


After drop: 16 columns remaining
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype']


### 3. Parse timestamp

`valid` is an ISO 8601 string. Parse to UTC-aware datetime.

In [11]:
stormreports = cleaner.parse_stormreport_timestamps(stormreports)
print("valid range:", stormreports["valid"].min(), "→", stormreports["valid"].max())

21:20:10  INFO     Parsed stormreports.valid timestamps


valid range: 2016-01-03 01:35:00+00:00 → 2025-12-29 12:42:00+00:00


### 4. Normalize `source`

Four issues addressed in order:

1. **Case and whitespace** — strip and uppercase collapses 132 → 91 unique values
2. **Truncation artifacts** — values where the leading character(s) were clipped (e.g. `MERGENCY MNGR`, `UBLIC`, `ROADCAST MEDIA`) mapped back to their canonical form
3. **Abbreviation variants** — `COUNTY EMERGY MG` / `COUNTY EMERGY MGMT` → `COUNTY OFFICIAL`; `NEWSPAPER` → `PRINT MEDIA`; `SOCIAL MEDIA` retained as distinct from `BROADCAST MEDIA` and `PRINT MEDIA`; automated station codes (`WEATHERFLOW`, `WXFLOW`, `MESOWEST`, etc.) → `AUTOMATED STATION`
4. **Null and junk values** (`X`, `NONE`, `UNKNOWN`, `FIRE`) — imputed as `"not_provided"`

In [12]:
stormreports = cleaner.normalize_source(stormreports)
print(f"Null source: {stormreports['source'].isnull().sum()}")
print()
print(stormreports["source"].value_counts().to_string())

21:20:11  INFO     Normalized source: 183 → 90 unique values


Null source: 0

source
PUBLIC                    91676
EMERGENCY MNGR            52971
911 CALL CENTER           46004
TRAINED SPOTTER           42777
LAW ENFORCEMENT           29525
BROADCAST MEDIA           17694
MESONET                   16653
NWS STORM SURVEY          10433
DEPT OF HIGHWAYS           9655
AMATEUR RADIO              9362
FIRE DEPT/RESCUE           8921
NWS EMPLOYEE               7131
ASOS                       6919
SOCIAL MEDIA               4712
UTILITY COMPANY            4652
STORM CHASER               4493
COUNTY OFFICIAL            3200
AWOS                       2854
CO-OP OBSERVER             2293
OTHER FEDERAL              1250
COCORAHS                   1185
PARK/FOREST SRVC            854
PRINT MEDIA                 826
OFFICIAL NWS OBS            795
TWITTER                     329
FACEBOOK                    277
POST OFFICE                 144
AUTOMATED STATION           139
not_provided                 95
NWS OFFICE                   45
SHERIFF OFFICE   

### 5. Inspect `leadtime`

`leadtime` is the API-supplied lead time in minutes between warning issuance and each storm report. It is already numeric in the extracted CSV — no parsing needed. Non-null for all warned reports; null for unwarned reports.

In [13]:
warned_sr = stormreports[stormreports["warned"] == True]
print(f"leadtime non-null: {warned_sr['leadtime'].notna().sum():,} of {len(warned_sr):,} warned rows")
print()
print(warned_sr["leadtime"].describe(percentiles=[.25, .5, .75, .90, .95, .99]).to_string())

leadtime non-null: 288,639 of 288,639 warned rows

count    288639.000000
mean         35.400836
std          70.568995
min           0.000000
25%          12.000000
50%          23.000000
75%          37.000000
90%          59.000000
95%         111.000000
99%         282.000000
max        7996.000000


### 6. Cap extreme `leadtime` values

`leadtime` is API-supplied, computed server-side as the difference in minutes between warning issuance and storm report time. FF warnings can spatially overlap storm reports from days later, producing physically implausible values (max ~7,996 min). TO and SV are naturally bounded. Cap per lsrtype at the 99th percentile; `leadtime_capped` flags affected rows.

In [14]:
stormreports = cleaner.cap_leadtime(stormreports)
print(f"Total capped: {stormreports['leadtime_capped'].sum():,}")
print("\nPost-cap distribution by lsrtype (warned only):")
warned_mask = stormreports["warned"] == True
print(
    stormreports[warned_mask]
    .groupby("lsrtype")["leadtime"]
    .describe(percentiles=[.25, .5, .75, .95])
    .to_string()
)

21:20:11  INFO     leadtime cap: FF cap=581 min, 427 rows capped
21:20:11  INFO     leadtime cap: SV cap=61 min, 2,299 rows capped
21:20:11  INFO     leadtime cap: TO cap=49 min, 105 rows capped


Total capped: 2,831

Post-cap distribution by lsrtype (warned only):
            count        mean         std  min   25%   50%    75%    95%    max
lsrtype                                                                        
FF        42836.0  105.462158  104.173427  0.0  35.0  74.0  140.0  316.0  581.0
SV       234462.0   22.713949   14.472581  0.0  11.0  21.0   33.0   49.0   61.0
TO        11341.0   15.278282   11.685185  0.0   6.0  13.0   23.0   38.0   49.0


### 7. Flag TDQ reports

`tdq` ("Too Difficult to Qualify") marks reports that NWS could not verify as a true hazardous event. These are retained in the dataset but flagged — downstream analyses that compute POD should consider excluding them to avoid inflating verified event counts.

In [15]:
tdq_count = stormreports["tdq"].sum()
print(f"TDQ reports: {tdq_count:,} of {len(stormreports):,} ({tdq_count/len(stormreports):.1%})")
print()
print("TDQ by lsrtype:")
print(stormreports.groupby("lsrtype")["tdq"].sum().to_string())

TDQ reports: 7,398 of 378,268 (2.0%)

TDQ by lsrtype:
lsrtype
FF      82
SV    7316
TO       0


### 8. Impute null `city` values

A small number of rows have no city. For each, reverse-geocode `lon0`/`lat0` via the Nominatim API and use the most specific named place available (city → town → village → hamlet), falling back to `"not_provided"` if the location is too rural to resolve.

In [16]:
print(f"Null city before: {stormreports['city'].isnull().sum()}")
stormreports = cleaner.impute_null_cities(stormreports)
print(f"Null city after:  {stormreports['city'].isnull().sum()}")

21:20:11  INFO     Imputing 5 null city values via Nominatim ...


Null city before: 5


21:20:12  INFO       city imputed: idx=115254 (43.1, -96.19) → 'Sioux Center'
21:20:13  INFO       city imputed: idx=115277 (44.12, -95.74) → 'not_provided'
21:20:15  INFO       city imputed: idx=139361 (34.29, -82.25) → 'Hodges'
21:20:16  INFO       city imputed: idx=139830 (34.35, -82.06) → 'Waterloo'
21:20:17  INFO       city imputed: idx=139841 (34.11, -82.25) → 'Promised Land'


Null city after:  0


### 9. Impute null `county` values

18 rows (all AJK 2020, southeast Alaska) have no county. Alaska uses boroughs rather than counties, and some areas are part of the Unorganized Borough — Nominatim may return `county`, `city` (for unified city-boroughs like Juneau), or nothing. Use the same reverse-geocode approach, falling back through `county` → `city` → `state_district` → `"not_provided"`.

In [17]:
print(f"Null county before: {stormreports['county'].isnull().sum()}")
stormreports = cleaner.impute_null_counties(stormreports)
print(f"Null county after:  {stormreports['county'].isnull().sum()}")

21:20:18  INFO     Imputing 51 null county values via Nominatim ...


Null county before: 51


21:20:19  INFO       county imputed: idx=5727 (61.81, -147.5) → 'Matanuska-Susitna Borough'
21:20:21  INFO       county imputed: idx=5728 (61.08, -146.3) → 'Unorganized Borough'
21:20:22  INFO       county imputed: idx=5729 (61.16, -149.9) → 'Anchorage'
21:20:23  INFO       county imputed: idx=5747 (63.77, -145.95) → 'Unorganized Borough'
21:20:32  INFO       county imputed: idx=5748 (63.44, -150.27) → 'Denali Borough'
21:20:33  INFO       county imputed: idx=5762 (55.21, -132.82) → 'Unorganized Borough'
21:20:34  INFO       county imputed: idx=5763 (57.06, -135.36) → 'Sitka'
21:20:36  INFO       county imputed: idx=5764 (57.05, -135.23) → 'Sitka'
21:20:37  INFO       county imputed: idx=5765 (57.12, -135.39) → 'Sitka'
21:20:39  INFO       county imputed: idx=5766 (58.28, -134.37) → 'Juneau'
21:20:40  INFO       county imputed: idx=5767 (58.3, -134.41) → 'Juneau'
21:20:42  INFO       county imputed: idx=5768 (59.43, -135.82) → 'Haines Borough'
21:20:43  INFO       county imputed: idx=5

Null county after:  0


### 10. Save cleaned storm reports

In [18]:
cleaner.save(stormreports, "stormreports.csv")
print(f"Saved {len(stormreports):,} rows × {len(stormreports.columns)} columns → data/03_cleaning/stormreports.csv")
print(stormreports.columns.tolist())

21:21:44  INFO     Saved 378,268 rows to ../data/03_cleaning/stormreports.csv


Saved 378,268 rows × 17 columns → data/03_cleaning/stormreports.csv
['wfo', 'year', 'valid', 'city', 'county', 'state', 'source', 'remark', 'typetext', 'lon0', 'lat0', 'events', 'tdq', 'warned', 'leadtime', 'lsrtype', 'leadtime_capped']


## Diagnostics

In [19]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 261557 entries, 0 to 261556
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype              
---  ------                 --------------   -----              
 0   wfo                    261557 non-null  str                
 1   year                   261557 non-null  int64              
 2   phenomena              261557 non-null  str                
 3   eventid                261557 non-null  int64              
 4   sharedborder           261544 non-null  float64            
 5   issue                  261557 non-null  datetime64[us, UTC]
 6   expire                 261557 non-null  datetime64[us, UTC]
 7   svr_tornado_possible   261557 non-null  bool               
 8   hailtag                220924 non-null  float64            
 9   windtag                195210 non-null  float64            
 10  carea                  261557 non-null  float64            
 11  parea                  261557 non-null  float64   

In [20]:
stormreports.info()

<class 'pandas.DataFrame'>
RangeIndex: 378268 entries, 0 to 378267
Data columns (total 17 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   wfo              378268 non-null  str                
 1   year             378268 non-null  int64              
 2   valid            378268 non-null  datetime64[us, UTC]
 3   city             378268 non-null  str                
 4   county           378268 non-null  str                
 5   state            378257 non-null  str                
 6   source           378268 non-null  str                
 7   remark           330140 non-null  str                
 8   typetext         378268 non-null  str                
 9   lon0             378268 non-null  float64            
 10  lat0             378268 non-null  float64            
 11  events           301658 non-null  str                
 12  tdq              378268 non-null  bool               
 13  warned    